# CASE/WHEN Functions in Databricks PySpark

The CASE function (implemented as `when()` and `otherwise()` in PySpark) provides conditional logic similar to SQL CASE statements or if-else blocks in programming.

## Core Functions

### 1. **when() and otherwise()**
Create conditional expressions to transform column values based on conditions.

**Syntax:**
```python
from pyspark.sql.functions import when, col

df.withColumn("new_column",
    when(condition1, value1)
    .when(condition2, value2)
    .otherwise(default_value)
)
```

### 2. **Basic Usage**
```python
from pyspark.sql.functions import when, col

# Simple if-else
df.withColumn("category",
    when(col("age") < 18, "Minor")
    .otherwise("Adult")
)

# Multiple conditions (if-elif-else)
df.withColumn("age_group",
    when(col("age") < 13, "Child")
    .when(col("age") < 20, "Teen")
    .when(col("age") < 65, "Adult")
    .otherwise("Senior")
)
```

## Common Patterns

### Pattern 1: Simple Binary Condition
```python
# Flag values based on threshold
df.withColumn("high_value",
    when(col("amount") > 1000, "Yes")
    .otherwise("No")
)
```

### Pattern 2: Multiple Conditions with Logical Operators
```python
# Combine conditions with & (AND) or | (OR)
df.withColumn("status",
    when((col("age") >= 18) & (col("verified") == True), "Active")
    .when((col("age") < 18) | (col("verified") == False), "Pending")
    .otherwise("Unknown")
)

# Important: Use & and | (not 'and'/'or'), and wrap conditions in parentheses
```

### Pattern 3: Nested CASE Statements
```python
# Nested conditions for complex logic
df.withColumn("tier",
    when(col("country") == "US",
        when(col("revenue") > 10000, "US Premium")
        .when(col("revenue") > 5000, "US Standard")
        .otherwise("US Basic")
    )
    .when(col("country") == "UK",
        when(col("revenue") > 8000, "UK Premium")
        .otherwise("UK Standard")
    )
    .otherwise("International")
)
```

### Pattern 4: Return Different Data Types
```python
# Return numeric values
df.withColumn("discount",
    when(col("customer_type") == "VIP", 0.20)
    .when(col("customer_type") == "Member", 0.10)
    .otherwise(0.0)
)

# Return boolean values
df.withColumn("eligible",
    when(col("score") >= 70, True)
    .otherwise(False)
)
```

### Pattern 5: NULL Handling
```python
from pyspark.sql.functions import isnull

# Check for NULL values
df.withColumn("clean_value",
    when(col("value").isNull(), 0)
    .otherwise(col("value"))
)

# Or using isnull function
df.withColumn("clean_value",
    when(isnull(col("value")), 0)
    .otherwise(col("value"))
)
```

### Pattern 6: String Matching
```python
from pyspark.sql.functions import col

# Using contains, startswith, endswith
df.withColumn("category",
    when(col("product_name").contains("Pro"), "Professional")
    .when(col("product_name").startswith("Basic"), "Entry Level")
    .otherwise("Standard")
)

# Using LIKE pattern (SQL-style)
df.withColumn("type",
    when(col("email").like("%@gmail.com"), "Gmail")
    .when(col("email").like("%@yahoo.com"), "Yahoo")
    .otherwise("Other")
)
```

### Pattern 7: Working with Arrays and Complex Types
```python
from pyspark.sql.functions import size, array_contains

# Check array properties
df.withColumn("has_items",
    when(size(col("items")) > 0, True)
    .otherwise(False)
)

# Check if array contains value
df.withColumn("has_premium",
    when(array_contains(col("features"), "premium"), "Yes")
    .otherwise("No")
)
```

## SQL Alternative

You can also use CASE directly in SQL queries:

```sql
SELECT
    name,
    CASE
        WHEN age < 13 THEN 'Child'
        WHEN age < 20 THEN 'Teen'
        WHEN age < 65 THEN 'Adult'
        ELSE 'Senior'
    END AS age_group
FROM table
```

## Important Notes

### Evaluation Order
* Conditions are evaluated **top to bottom**
* First matching condition wins (like if-elif-else)
* Remaining conditions are **not evaluated** after a match

### Performance Tips
* **Order matters**: Put most common conditions first for better performance
* **Avoid redundant checks**: Once a condition matches, evaluation stops
* **Use column expressions**: Avoid UDFs when `when()` can handle the logic

### Common Mistakes

❌ **Wrong: Using 'and'/'or' operators**
```python
# This will cause errors
when((col("a") > 5) and (col("b") < 10), "value")
```

✅ **Correct: Using & and | with parentheses**
```python
when((col("a") > 5) & (col("b") < 10), "value")
```

❌ **Wrong: Missing otherwise() with NULL result**
```python
# If no condition matches, result is NULL
when(col("status") == "active", 1)
```

✅ **Correct: Always provide otherwise() for complete coverage**
```python
when(col("status") == "active", 1)
.otherwise(0)
```

## Comparison with SQL

| PySpark                                          | SQL                                  |
| ------------------------------------------------ | ------------------------------------ |
| `when(condition, value).otherwise(default)`      | `CASE WHEN condition THEN value ELSE default END` |
| `&` (and), `\|` (or), `~` (not)                  | `AND`, `OR`, `NOT`                   |
| `col("name").isNull()`                           | `name IS NULL`                       |
| `col("name").isNotNull()`                        | `name IS NOT NULL`                   |
| `col("name").isin(["A", "B"])`                  | `name IN ('A', 'B')`                 |

## Use Cases

* **Data cleaning**: Replace invalid values, handle NULLs
* **Feature engineering**: Create categorical variables, bins, flags
* **Business logic**: Apply pricing rules, eligibility checks, status calculations
* **Data transformation**: Standardize values, map codes to descriptions
* **Conditional aggregation**: Combined with groupBy for conditional metrics

### Create a sample dataframe

In [0]:
data_student = [
   ("raja", "science", 80, "p", 90),
   ("rakesh", "maths", 90, "p", 70),
   ("rama", "english", 20, "f", 80),
   ("ramesh", "science", 30, "f", 50),
   ("raju", "maths", 40, "f", 60),
   ("ramu", "english", 35, "f", 70),
   ("rajesh", "science", 60, "P", 80),
   ("raj", "maths", 70, "P", 90),
   ("ravi", "english", None, "f", 100)
   ]

schema = 'name string, subject string, marks int, status string, attendance int'

df = spark.createDataFrame(data=data_student, schema=schema)
  
df.show()

df.printSchema()

### Update the existing column

In [0]:
from pyspark.sql.functions import when, col, lit, expr

In [0]:
df1 = df.withColumn("status", when(col("marks") >= 50, lit("pass"))
                             .when(col("marks") < 50, lit("fail"))
                             .otherwise(lit("absent"))
)

df1.show()

### Create a New Column

In [0]:
df.withColumn("new_status", when(col("marks") >= 50, lit("pass"))
                             .when(col("marks") < 50, lit("fail"))
                             .otherwise(lit("absent"))).show()

In [0]:
df.withColumn("new_status", expr("case when marks >= 50 then 'pass' \
                                    when marks < 50 then 'fail'\
                                    else 'absent' end")).show()

### Multi-Condition using AND and OR operators

In [0]:
df.withColumn("grade", when((col("marks") >= 80) & (col("attendance") >= 80), "distinction")\
                      .when((col("marks") >= 60) & (col("attendance") >= 60), "first")\
                      .when((col("marks") >= 50) & (col("attendance") >= 50), "second")\
                      .otherwise("average")).show()